In [1]:
import requests, pandas as pd, time, re, os
import urllib3
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

urllib3.disable_warnings()

BASE_URL = "https://www.juntadeandalucia.es/institutodeestadisticaycartografia/intranet/admin/rest/v1.0"
HEADERS = {"Accept": "application/json", "User-Agent": "Mozilla/5.0"}
RUTA_BASE = r"C:\Users\JorgeB\Desktop\Proyecto-DA\badea_datos_andalucia"

secciones = {
    "1": "1_entorno_fisico_medio_ambiente",
    "2": "2_demografia_poblacion",
    "3": "3_sociedad",
    "4": "4_economia",
    "5": "5_mercado_trabajo",
    "6": "6_hacienda"
}

# 1. ÍNDICE
print("🔍 Obteniendo índice...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.juntadeandalucia.es/institutodeestadisticaycartografia/badea/informe/anual?CodOper=b3_151&idNode=23204")
time.sleep(8)
soup = BeautifulSoup(driver.page_source, "html.parser")
driver.quit()

consultas = []
for a in soup.find_all("a", href=True):
    m = re.search(r"consulta/anual/(\d+)", a["href"])
    if m:
        consultas.append({"id": int(m.group(1)), "titulo": a.text.strip()})
consultas_unicas = list({c["id"]: c for c in consultas}.values())
print(f"✅ {len(consultas_unicas)} consultas")

# 2. SALTAR YA DESCARGADAS
ya_descargados = set()
for root, dirs, files in os.walk(RUTA_BASE):
    for f in files:
        if f.endswith(".parquet"):
            ya_descargados.add(int(f.split("_")[0]))
pendientes = [c for c in consultas_unicas if c['id'] not in ya_descargados]
print(f"⏭️ Ya descargadas: {len(ya_descargados)} | Pendientes: {len(pendientes)}")

# 3. FUNCIONES
def get_carpeta(titulo):
    m = re.match(r'^(\d+)\.(\d+)\.?', titulo)
    if m:
        sec, subsec = m.group(1), m.group(2)
        return os.path.join(RUTA_BASE, secciones.get(sec, f"{sec}_otros"), f"{sec}.{subsec}")
    return os.path.join(RUTA_BASE, "otros")

def obtener_años(consulta_id):
    try:
        r = requests.get(f"{BASE_URL}/jerarquia/2?consultaId={consulta_id}&alias=D_TEMPORAL_0", headers=HEADERS, verify=False, timeout=30)
        return [{"id": h["id"], "cod": h["cod"]} for h in r.json()["data"]["children"]
                if h["cod"].isdigit() and 1990 <= int(h["cod"]) <= 2025]
    except:
        return []

# 4. DESCARGA - para con ⬛, continúa relanzando la celda
errores = []
print(f"🚀 Iniciando descarga...\n")

for i, c in enumerate(pendientes):
    consulta_id, titulo = c['id'], c['titulo']
    print(f"[{i+1}/{len(pendientes)}] {consulta_id} - {titulo[:45]}", end=" → ")
    try:
        años = obtener_años(consulta_id)
        filas = []
        if años:
            for a in años:
                try:
                    r = requests.get(f"{BASE_URL}/consulta/{consulta_id}", params={"D_TEMPORAL_0": str(a["id"])}, headers=HEADERS, verify=False, timeout=30)
                    data = r.json()
                    if not data.get("data"): continue
                    medida = data["measures"][0]["des"]
                    for fila in data["data"]:
                        reg = {data["hierarchies"][j]["des"]: fila[j]["des"] for j in range(len(data["hierarchies"]))}
                        reg[medida] = fila[-1]["val"]
                        filas.append(reg)
                except:
                    pass
                time.sleep(0.05)
        else:
            r = requests.get(f"{BASE_URL}/consulta/{consulta_id}", headers=HEADERS, verify=False, timeout=30)
            data = r.json()
            if data.get("data"):
                medida = data["measures"][0]["des"]
                for fila in data["data"]:
                    reg = {data["hierarchies"][j]["des"]: fila[j]["des"] for j in range(len(data["hierarchies"]))}
                    reg[medida] = fila[-1]["val"]
                    filas.append(reg)

        if not filas:
            print("⚠️ vacío")
            errores.append(c)
            continue

        df = pd.DataFrame(filas)
        carpeta = get_carpeta(titulo)
        os.makedirs(carpeta, exist_ok=True)
        nombre = f"{consulta_id}_{titulo[:40].replace(' ','_').replace('/','_').replace('.','')}.parquet"
        df.to_parquet(os.path.join(carpeta, nombre), index=False)
        print(f"✅ {len(df)} filas")

    except Exception as e:
        print(f"❌ {e}")
        errores.append(c)
    time.sleep(0.1)

print(f"\n🏁 Sesión terminada. Errores: {len(errores)}")
print(f"📦 Total descargadas: {len(ya_descargados) + len(pendientes) - len(errores)}")

🔍 Obteniendo índice...
✅ 454 consultas
⏭️ Ya descargadas: 113 | Pendientes: 341
🚀 Iniciando descarga...

[1/341] 22874 - 2.3.2.2.7. Inmigraciones procedentes del extr → ✅ 54672 filas
[2/341] 22971 - 2.3.2.2.8. Inmigraciones procedentes del extr → ✅ 273360 filas
[3/341] 22974 - 2.3.2.2.9. Inmigraciones procedentes del extr → ✅ 164016 filas
[4/341] 36361 - 2.4.1. Edad media de la población por sexo (P → ✅ 54891 filas
[5/341] 36362 - 2.4.2. Edad media de la población por naciona → ✅ 57945 filas
[6/341] 51752 - 2.4.3. Índice de dependencia → ✅ 60147 filas
[7/341] 51744 - 2.4.4. Índice de envejecimiento → ❌ Expecting value: line 1 column 1 (char 0)
[8/341] 19064 - 3.1.1.1.1. Centros públicos por nivel educati → ❌ Expecting value: line 1 column 1 (char 0)
[9/341] 18417 - 3.1.1.1.2. Centros públicos de adultos → ❌ Expecting value: line 1 column 1 (char 0)
[10/341] 18943 - 3.1.1.2.1. Centros privados concertados por n → ❌ Expecting value: line 1 column 1 (char 0)
[11/341] 51470 - 3.1.1.2.2. Ce